# BUNAGUARD: A Coffee Leaf Disease Research Prototype for Ugandan Smallholder Farmers

In [ ]:
# Core Python
from pathlib import Path
# Data and images
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
# Dataset splitting and evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
# Grad-CAM
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
# Demo interface
import gradio as gr

In [ ]:
# Reproducibility
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED)
# Separate generators initialized with the same seed so baseline and augmented
# DataLoaders see identical shuffle sequences. Note: this controls shuffle order
# only — the two models still randomly initialize separate classification heads,
# so the baseline-vs-augmented comparison is not perfectly controlled.
g_baseline = torch.Generator().manual_seed(SEED)
g_aug = torch.Generator().manual_seed(SEED)

In [ ]:
#build transform pipeline
my_transform=transforms.Compose([
transforms.Resize((224,224)),
transforms.ToTensor(),
transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

In [ ]:
#build the catalog
project_root = Path.cwd().parent
dataset_path = (project_root / "data" / "prepared" / "coffee_leaf_dataset")
coffee_dataset = datasets.ImageFolder(
    root=dataset_path
)


In [ ]:
#inspect
print("total image:",len(coffee_dataset))
print("classes:",coffee_dataset.classes)
print("class_labels:",coffee_dataset.class_to_idx)

In [ ]:
#split the dataset
all_indices=list(range(len(coffee_dataset)))
all_labels_split=coffee_dataset.targets
train_indices,temp_indices=train_test_split(
    all_indices,
    test_size=0.3,
    random_state=42,
    stratify=all_labels_split
)
temp_labels=[all_labels_split[i] for i in temp_indices]
val_indices,test_indices=train_test_split(
    temp_indices,
    test_size=0.5,
    random_state=42,
    stratify=temp_labels
)

In [ ]:
#lets check
print(len(val_indices))
print(len(train_indices))
print(len(test_indices))


In [ ]:
#dataloading
#first lets create transformed catalog
baseline_catalog=datasets.ImageFolder(
    root=dataset_path,
    transform=my_transform
)
train_dataset=Subset(baseline_catalog,train_indices)
val_dataset=Subset(baseline_catalog,val_indices)
test_dataset=Subset(baseline_catalog,test_indices)
#now i can create dataloader since i have already created baseline catalog before augmentation
train_data_loader=DataLoader(train_dataset,batch_size=32,shuffle=True,generator=g_baseline)
val_data_loader=DataLoader(val_dataset,batch_size=32,shuffle=False)
test_data_loader=DataLoader(test_dataset,batch_size=32,shuffle=False)
# Sanity check this advances g_baseline's internal state.
image,labels=next(iter(train_data_loader))
print(image.shape)
print(labels.shape)
print(len(test_dataset))
# Reseed g_baseline so that, when baseline training begins, it sees the SAME
# shuffle sequence as g_aug (which has not been consumed yet).
g_baseline.manual_seed(SEED)

In [ ]:
#download pretrained  model
model=models.resnet18(weights="IMAGENET1K_V1")
#freez teh backbone
for param in model.parameters():
    param.requires_grad=False
#replace the head
model.fc=nn.Linear(model.fc.in_features,len(coffee_dataset.classes))
print(model.fc)

In [ ]:
#device selection
if torch.cuda.is_available():
    device="cuda"
elif torch.backends.mps.is_available():
    device="mps"
else:
    device="cpu"
model=model.to(device)
print(device)

In [ ]:
#build up loss and optimizer
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.fc.parameters(),lr=0.001)

### Current experiment status

The baseline model completed all 10 training epochs. The best-validation checkpoint reached 92.25% validation accuracy and produced 89.71% accuracy on the held-out test set.

In [ ]:
EPOCHS=10
train_loss=[]
train_accuracies=[]
val_accuracies=[]
best_val_acc=0.0
for epoch in range(EPOCHS):
    model.train()
    running_loss=0.0
    train_correct=0
    train_total=0
    for images,labels in train_data_loader:
        images=images.to(device)
        labels=labels.to(device)
        #forward
        output=model(images)
        #compute the loss
        loss=criterion(output,labels)
        #backward
        optimizer.zero_grad()
        loss.backward()
        #optimizer step updates the classification head
        optimizer.step()
        running_loss+=loss.item()
        _,predicted=torch.max(output,1)
        train_correct+=(predicted==labels).sum().item()
        train_total+=labels.size(0)
    avg_train_loss=running_loss/len(train_data_loader)
    train_loss.append(avg_train_loss)
    train_acc=100*train_correct/train_total
    train_accuracies.append(train_acc)
    #validation phase
    model.eval()
    correct=0
    total=0
    with torch.no_grad():
        for images,labels in val_data_loader:
            images=images.to(device)
            labels=labels.to(device)
            output=model(images)
            _,predicted=torch.max(output,1)
            correct+=(predicted==labels).sum().item()
            total+=labels.size(0)
    val_accuracy=100*correct/total
    val_accuracies.append(val_accuracy)
    # Save the best validation checkpoint inline so it survives interruption
    if val_accuracy > best_val_acc:
        best_val_acc = val_accuracy
        torch.save(model.state_dict(), "baseline_model.pth")
    print(f"Epoch {epoch+1}/{EPOCHS}  Loss: {avg_train_loss:.4f}  Train Acc: {train_acc:.2f}%  Val Acc: {val_accuracy:.2f}%")

# Reload best validation weights so evaluation, Grad-CAM, and Gradio use them
model.load_state_dict(
    torch.load("baseline_model.pth", map_location=device, weights_only=True)
)
print(f"Best validation accuracy: {best_val_acc:.2f}% — best checkpoint reloaded into model.")

In [ ]:
#evaluation phase
model.eval()
all_pred=[]
all_labels=[]
with torch.no_grad():
    for images,labels in test_data_loader:
        images=images.to(device)
        labels=labels.to(device)
        output=model(images)
        _,predicted=torch.max(output,1)
        all_pred.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
class_names=coffee_dataset.classes
accuracy=accuracy_score(all_labels,all_pred)
baseline_accuracy = accuracy * 100
cm=confusion_matrix(all_labels,all_pred)
report=classification_report(all_labels,all_pred,target_names=class_names,zero_division=0)
print(f"baseline accuracy: {baseline_accuracy:.2f}%")
print(report)

# Prominent per class precision / recall / F1 — important because phoma is the minority class
report_dict = classification_report(
    all_labels, all_pred,
    target_names=class_names,
    output_dict=True, zero_division=0
)
print("\nPer-class metrics (baseline):")
for cls in class_names:
    m = report_dict[cls]
    print(f"  {cls:12s}  precision={m['precision']:.3f}  "
          f"recall={m['recall']:.3f}  f1={m['f1-score']:.3f}")

In [ ]:
#import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Baseline (No Augmentation)')
plt.tight_layout()
plt.savefig('baseline_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
#Training curves plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, EPOCHS+1), train_loss, marker='o')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Baseline Training Loss')

ax2.plot(range(1, EPOCHS+1), train_accuracies, marker='o', label='Train')
ax2.plot(range(1, EPOCHS+1), val_accuracies, marker='s', color='green', label='Validation')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Baseline Train vs Validation Accuracy')
ax2.legend()

plt.tight_layout()
plt.savefig('baseline_training_curves.png', dpi=150)
plt.show()
# Best-validation checkpoint was already saved inline during training,
# so no torch.save here — that would overwrite the best with final-epoch weights.
print("Baseline best checkpoint already saved to baseline_model.pth during training.")

In [ ]:
# Training transform with augmentation
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.GaussianBlur(kernel_size=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [ ]:
#augmented catalog and dataloader
aug_train_catalog = datasets.ImageFolder(
    root=dataset_path,
    transform=train_transform
)
aug_train_dataset = Subset(aug_train_catalog, train_indices)
aug_train_loader = DataLoader(aug_train_dataset, batch_size=32, shuffle=True, generator=g_aug)
print(f"Augmented train: {len(aug_train_dataset)}")

In [ ]:
#model for augmentation
model_aug = models.resnet18(weights="IMAGENET1K_V1")
for param in model_aug.parameters():
    param.requires_grad = False
model_aug.fc = nn.Linear(model_aug.fc.in_features, len(coffee_dataset.classes))
model_aug = model_aug.to(device)
#loss and optimizer
criterion_aug = nn.CrossEntropyLoss()
optimizer_aug = optim.Adam(model_aug.fc.parameters(), lr=0.001)

In [ ]:
EPOCHS = 10
aug_train_losses = []
aug_train_accuracies = []
aug_val_accuracies = []
best_val_acc_aug = 0.0
for epoch in range(EPOCHS):
    model_aug.train()
    running_loss = 0.0
    train_correct = 0
    train_total = 0
    for images, labels in aug_train_loader:
        images=images.to(device)
        labels=labels.to(device)
        #forward
        output =model_aug(images)
        #loss
        loss=criterion_aug(output, labels)
        #backward
        optimizer_aug.zero_grad()
        loss.backward()
        #optimizer step
        optimizer_aug.step()
        running_loss += loss.item()
        _, predicted = torch.max(output, 1)
        train_correct += (predicted == labels).sum().item()
        train_total += labels.size(0)
    avg_loss = running_loss / len(aug_train_loader)
    aug_train_losses.append(avg_loss)
    train_acc = 100 * train_correct / train_total
    aug_train_accuracies.append(train_acc)
    model_aug.eval()
    correct=0
    total =0
    with torch.no_grad():
        for images,labels in val_data_loader:
            images=images.to(device)
            labels=labels.to(device)
            output=model_aug(images)
            _, predicted=torch.max(output, 1)
            correct += (predicted == labels).sum().item()
            total+=labels.size(0)
    val_acc = 100 * correct / total
    aug_val_accuracies.append(val_acc)
    # Save best-validation checkpoint inline
    if val_acc > best_val_acc_aug:
        best_val_acc_aug = val_acc
        torch.save(model_aug.state_dict(), "augmented_model.pth")
    print(f"Epoch {epoch+1}/{EPOCHS}  Loss: {avg_loss:.4f}  Train Acc: {train_acc:.2f}%  Val Acc: {val_acc:.2f}%")

# Reload best-validation weights into model_aug
model_aug.load_state_dict(
    torch.load("augmented_model.pth", map_location=device, weights_only=True)
)
print(f"Best augmented validation accuracy: {best_val_acc_aug:.2f}% — best checkpoint reloaded into model_aug.")

In [ ]:
# augmented evaluation
model_aug.eval()
aug_preds=[]
aug_labels_list =[]
with torch.no_grad():
    for images, labels in test_data_loader:
        images=images.to(device)
        labels=labels.to(device)
        output=model_aug(images)
        _, predicted=torch.max(output, 1)
        aug_preds.extend(predicted.cpu().numpy())
        aug_labels_list.extend(labels.cpu().numpy())
augmented_accuracy = accuracy_score(aug_labels_list, aug_preds) * 100
print(f"BASELINE accuracy:    {baseline_accuracy:.2f}%")
print(f"AUGMENTED accuracy:   {augmented_accuracy:.2f}%")
print(f"Improvement:          {augmented_accuracy - baseline_accuracy:+.2f}%")
# Best augmented checkpoint already saved inline during training (no torch.save here).
print("Augmented best checkpoint already saved to augmented_model.pth during training.")

### Note on the augmentation comparison

The results above come from one seeded training run per model. The same split, seed, and DataLoader shuffle sequence were used, which improves reproducibility, but each model still received a separately initialized classification head.

The baseline model achieved 89.71% test accuracy, while the augmented model achieved 90.07%, a difference of only +0.37 percentage points. This small difference is not enough to conclude that augmentation improved performance.

The baseline model remains the model used by the demo because model selection was based on validation performance, where it achieved 92.25% compared with the augmented model's 90.77%. The test set was kept for final evaluation rather than used to choose the model. A stronger conclusion about augmentation would require multiple seeds and statistical testing.

In [ ]:
# Augmented confusion matrix + report
class_names = coffee_dataset.classes
cm_aug = confusion_matrix(aug_labels_list, aug_preds)
report_aug = classification_report(aug_labels_list, aug_preds, target_names=class_names, zero_division=0)

print(f"Augmented Accuracy: {augmented_accuracy:.2f}%")
print("\n", report_aug)

# Per class metrics for the augmented model, parallel to baseline
report_dict_aug = classification_report(
    aug_labels_list, aug_preds,
    target_names=class_names,
    output_dict=True, zero_division=0
)
print("Per-class metrics (augmented):")
for cls in class_names:
    m = report_dict_aug[cls]
    print(f"  {cls:12s}  precision={m['precision']:.3f}  "
          f"recall={m['recall']:.3f}  f1={m['f1-score']:.3f}")

plt.figure(figsize=(8, 6))
sns.heatmap(cm_aug, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Augmented')
plt.tight_layout()
plt.savefig('augmented_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
#Side by side training curves (baseline vs augmented):
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, EPOCHS+1), train_loss, marker='o', label='Baseline')
ax1.plot(range(1, EPOCHS+1), aug_train_losses, marker='s', label='Augmented')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss Comparison')
ax1.legend()

ax2.plot(range(1, EPOCHS+1), val_accuracies, marker='o', label='Baseline')
ax2.plot(range(1, EPOCHS+1), aug_val_accuracies, marker='s', label='Augmented')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Validation Accuracy Comparison')
ax2.legend()

plt.tight_layout()
plt.savefig('comparison_curves.png', dpi=150)
plt.show()

In [ ]:
!pip install grad-cam

In [ ]:
import numpy as np
from PIL import Image
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
# pick one test image per class.
# prefer correctly classified ones — otherwise grad-cam explains the wrong prediction.
def pick_test_example_for_class(target_class):
    model.eval()
    first_path = None
    first_pred = None
    for i in test_indices:
        if coffee_dataset.targets[i] != target_class:
            continue
        path = coffee_dataset.imgs[i][0]
        img = Image.open(path).convert("RGB")
        x = my_transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            pred = model(x).argmax(dim=1).item()
        if pred == target_class:
            return path, pred, True
        if first_path is None:
            first_path = path
            first_pred = pred
    if first_path is not None:
        return first_path, first_pred, False
    return None, None, None

def show_gradcam(image_path, true_class_name, pred_class_name, correct):
    model.eval()
    img = Image.open(image_path).convert("RGB")
    resized_img = img.resize((224, 224))
    img_array = np.array(resized_img).astype(np.float32) / 255.0
    x = my_transform(img).unsqueeze(0).to(device)

    # Temporarily enable gradients on layer4 so Grad-CAM can backpropagate.
    # We turn this back off after to keep the backbone frozen permanently.
    for param in model.layer4.parameters():
        param.requires_grad = True
    cam = GradCAM(
        model=model,
        target_layers=[model.layer4[-1]]
    )
    heatmap = cam(input_tensor=x)[0]
    visualization = show_cam_on_image(
        img_array,
        heatmap,
        use_rgb=True
    )
    for param in model.layer4.parameters():
        param.requires_grad = False

    status = "correct" if correct else "MISCLASSIFIED"
    title = (f"Grad-CAM (baseline): true={true_class_name}  "
             f"pred={pred_class_name}  [{status}]")
    plt.figure()
    plt.imshow(visualization)
    plt.title(title)
    plt.axis("off")
    out_name = f"gradcam_{true_class_name}.png"
    plt.savefig(out_name, dpi=150, bbox_inches="tight")
    plt.show()

# Show one example per class from the test set, preferring correctly-classified ones
for class_index, class_name in enumerate(coffee_dataset.classes):
    sample_path, pred_index, correct = pick_test_example_for_class(class_index)
    if sample_path is None:
        continue
    pred_name = coffee_dataset.classes[pred_index]
    print(f"Grad-CAM for true class '{class_name}' — predicted '{pred_name}' "
          f"({'correct' if correct else 'misclassified'}): {Path(sample_path).name}")
    show_gradcam(sample_path, class_name, pred_name, correct)

In [ ]:
# Out-of-distribution robustness test
# Probe the baseline model with a small convenience sample of inputs that are
# NOT coffee leaves 15 random real world photographs and 5 synthetic patterns.
# This exploratory test measures how often the 60% threshold rejects these inputs
# it is not a representative OOD benchmark or proof of deployment safety.
from pathlib import Path

ood_dir = project_root / "non_coffee_test"
threshold = 0.60

if not ood_dir.exists():
    print(f"⚠️  Folder not found: {ood_dir}")
    print("Create the folder and place 15-20 non-coffee images (jpg/png) inside, then re-run this cell.")
else:
    image_paths = sorted([p for p in ood_dir.iterdir()
                          if p.suffix.lower() in (".jpg", ".jpeg", ".png")])

    if not image_paths:
        print(f"⚠️  No images found in {ood_dir}. Add some images and re-run.")
    else:
        model.eval()
        refused = 0
        confident_wrong = 0
        rows = []

        with torch.no_grad():
            for path in image_paths:
                try:
                    img = Image.open(path).convert("RGB")
                except Exception as e:
                    rows.append((path.name, "ERROR", 0.0, f"could not open ({e})"))
                    continue
                x = my_transform(img).unsqueeze(0).to(device)
                probs = torch.softmax(model(x), dim=1)[0]
                top_prob, top_idx = torch.max(probs, dim=0)
                top_class = class_names[top_idx.item()]
                conf = top_prob.item()
                if conf < threshold:
                    refused += 1
                    status = "REFUSED (safe)"
                else:
                    confident_wrong += 1
                    status = "CONFIDENTLY WRONG"
                rows.append((path.name, top_class, conf, status))

        n = len(image_paths)
        print(f"OOD robustness test — {n} non-coffee images, threshold = {threshold:.2f}")
        print("=" * 78)
        print(f"{'image':<32} {'top class':<12} {'conf':>7}  status")
        print("-" * 78)
        for name, cls, conf, status in rows:
            short = (name[:29] + "...") if len(name) > 32 else name
            print(f"{short:<32} {cls:<12} {conf*100:>6.1f}%  {status}")
        print("-" * 78)
        print(f"\nSummary:")
        print(f"  Refused (safe behavior):       {refused}/{n} ({100*refused/n:.0f}%)")
        print(f"  Confidently wrong (failure):   {confident_wrong}/{n} ({100*confident_wrong/n:.0f}%)")

### Interpretation of the exploratory OOD test

The test used a small convenience sample of 20 non-coffee inputs: 15 random general-purpose photographs and 5 synthetic patterns. At the 60% confidence threshold, BunaGuard rejected 7/20 inputs (35%) and accepted 13/20 (65%) as one of its three coffee-leaf classes.

- Random real-world photographs: rejected 6/15 (40%); accepted 9/15 (60%).
- Synthetic patterns: rejected 1/5 (20%); accepted 4/5 (80%).

This result demonstrates that the current softmax
confidence threshold is not a reliable unknown-input detector. It does not measure performance on representative farmer uploads, because the sample is small, was not collected from Ugandan farmers, and mixes real photographs with synthetic patterns. A larger pre defined field-relevant OOD dataset and a dedicated OOD detection method are required before i deploym

# limitations of the demo

- only 3 coffee diseases covered. wilt, berry disease, cercospora, miner — not detected.
- the 60% threshold doesn't reliably reject non-coffee images.
the system assumes the user is uploading a coffee leaf on purpose.
- english covers many ugandan users (it's an official language). reaching rural non-english speakers would need luganda its my future work

In [ ]:
!pip install gradio

In [ ]:
# Always load the saved best-validation checkpoint before launching the demo.
model.load_state_dict(
    torch.load("baseline_model.pth", map_location=device, weights_only=True)
)
model.eval()


def check_image_quality(img):
    img_array = np.asarray(img.convert("RGB"))
    brightness = img_array.mean()
    height, width = img_array.shape[:2]

    if height < 50 or width < 50:
        return False, (
            "⚠️ ፎቶው በጣም ትንሽ ነው። ቅጠሉን በቅርብ ያንሱ።\n"
            "The image is too small. Take a closer photo of the leaf."
        )

    if brightness < 30:
        return False, (
            "⚠️ ፎቶው በጣም ጨለማ ነው። በብርሃን ቦታ እንደገና ያንሱ።\n"
            "The image is too dark. Take another photo in better lighting."
        )

    if brightness > 240:
        return False, (
            "⚠️ ፎቶው በጣም ብሩህ ነው። ጥላ ውስጥ እንደገና ያንሱ።\n"
            "The image is too bright. Take another photo in the shade."
        )

    return True, ""


all_questions = [
    "🔍 ይህ ምን አይነት በሽታ ነው? — What disease is this?",
    "🔍 ቅጠሉ ጤናማ ነው? — Is the leaf healthy?",
    "🔍 ቅጠሉ ላይ ያለው ነጠብጣብ ምንድን ነው? — What are these spots?",
    "🔍 ችግሩ ምን ያህል ከባድ ነው? — How severe is the problem?",
    "💊 ምን ላድርግ? — What should I do?",
    "💊 ምን መድሃኒት ልጠቀም? — What treatment should I use?",
    "💊 መቼ መርጨት አለብኝ? — When should treatment be applied?",
    "💊 የተበከሉ ቅጠሎችን ማስወገድ አለብኝ? — Should damaged leaves be removed?",
    "🛡️ እንዴት መከላከል እችላለሁ? — How can I prevent it?",
    "🛡️ ጥንቃቄ ምን ማድረግ አለብኝ? — What precautions should I take?",
    "🛡️ ሌሎች ተክሎች ይበከላሉ? — Can it spread to other plants?",
    "🛡️ በየስንት ጊዜ ማረጋገጥ አለብኝ? — How often should I inspect?",
    "💧 ውሃ መቼ ማጠጣት አለብኝ? — When should I water?",
    "💧 ምን ያህል ውሃ ያስፈልጋል? — How much water is needed?",
    "💧 ከመጠን በላይ ውሃ ማጠጣት ችግር አለው? — Is overwatering harmful?",
    "💧 የመስኖ ዘዴ ምን ይመከራል? — Which irrigation method is recommended?",
    "🌿 ምርቴን ይጎዳል? — Will it affect production?",
    "🌿 የግብርና ባለሙያ ማማከር አለብኝ? — Should I consult an expert?",
    "🌿 ምን ዓይነት ማዳበሪያ ልጠቀም? — Which fertilizer should I use?",
    "🌿 ቡናዬ ለምን ቢጫ ሆነ? — Why is my coffee leaf yellow?",
]


intents = [
    "diagnosis", "diagnosis", "diagnosis", "severity",
    "treatment", "treatment", "treatment", "treatment",
    "prevention", "prevention", "spread", "inspection",
    "watering", "watering", "watering", "watering",
    "yield", "expert", "fertilizer", "yellowing"
]

question_to_intent = dict(zip(all_questions, intents))


amharic_names = {
    "healthy": "ጤናማ",
    "leaf_rust": "የቡና ቅጠል ዝገት",
    "phoma": "ፎማ"
}

english_names = {
    "healthy": "Healthy",
    "leaf_rust": "Coffee Leaf Rust",
    "phoma": "Phoma"
}


amharic_advice = {
    ("healthy", "diagnosis"): "ቅጠሉ ጤናማ ይመስላል። በየጊዜው መመርመርዎን ይቀጥሉ።",
    ("healthy", "treatment"): "ቅጠሉ ጤናማ ስለሚመስል ህክምና አያስፈልገውም።",
    ("healthy", "prevention"): "ቅጠሎችን በየጊዜው ይመልከቱ እና የወደቁ ቅጠሎችን ያጽዱ።",
    ("healthy", "watering"): "የውሃ መጠን በአፈርና በአየር ሁኔታ ይለያያል። ከመጠን በላይ አያጠጡ።",

    ("leaf_rust", "diagnosis"): "ቅጠሉ የቡና ቅጠል ዝገት ምልክቶች ያሉት ይመስላል።",
    ("leaf_rust", "treatment"): "የተጎዱ ቅጠሎችን ለይተው ያስወግዱ። ህክምና ከመጠቀምዎ በፊት ባለሙያን ያማክሩ።",
    ("leaf_rust", "prevention"): "ቅጠሎችን ይመርምሩ፣ የአየር ዝውውርን ያሻሽሉ እና የወደቁ ቅጠሎችን ያጽዱ።",
    ("leaf_rust", "watering"): "ቅጠሎቹን ሳያረጥቡ ተክሉን ከሥሩ ያጠጡ።",

    ("phoma", "diagnosis"): "ቅጠሉ የፎማ በሽታ ምልክቶች ያሉት ይመስላል።",
    ("phoma", "treatment"): "የተጎዱ ቅጠሎችን ለይተው ያስወግዱ። ህክምና ከመጠቀምዎ በፊት ባለሙያን ያማክሩ።",
    ("phoma", "prevention"): "የአየር ዝውውርን ለማሻሻል ቅርንጫፎችን ይከርክሙ።",
    ("phoma", "watering"): "ቅጠሎቹን ሳያረጥቡ ተክሉን ከሥሩ ያጠጡ።",
}


english_advice = {
    ("healthy", "diagnosis"): "The coffee leaf appears healthy. Continue regular inspection.",
    ("healthy", "treatment"): "No treatment appears necessary.",
    ("healthy", "prevention"): "Continue regular inspection and remove fallen leaves.",
    ("healthy", "watering"): "Water requirements depend on soil and weather. Avoid overwatering.",

    ("leaf_rust", "diagnosis"): "The leaf appears to show symptoms of coffee leaf rust.",
    ("leaf_rust", "treatment"): "Separate damaged leaves and consult a local agricultural extension officer or qualified agricultural expert before treatment.",
    ("leaf_rust", "prevention"): "Inspect leaves regularly, improve airflow, and remove fallen leaves.",
    ("leaf_rust", "watering"): "Water near the roots and avoid wetting the leaves.",

    ("phoma", "diagnosis"): "The leaf appears to show symptoms of Phoma disease.",
    ("phoma", "treatment"): "Separate damaged leaves and consult a local agricultural extension officer or qualified agricultural expert before treatment.",
    ("phoma", "prevention"): "Improve airflow by carefully pruning surrounding branches.",
    ("phoma", "watering"): "Water near the roots and avoid wetting the leaves.",
}


special_advice = {
    "severity": (
        "BunaGuard ከአንድ ፎቶ የበሽታውን ክብደት በትክክል መለካት አይችልም።",
        "BunaGuard cannot accurately measure disease severity from one image."
    ),
    "spread": (
        "በአቅራቢያ ያሉ የቡና ተክሎችን ይመርምሩ።",
        "Inspect nearby coffee plants for similar symptoms."
    ),
    "inspection": (
        "ቅጠሎችን በየጊዜው ይመርምሩ።",
        "Inspect coffee leaves regularly for early disease detection."
    ),
    "yield": (
        "የቅጠል በሽታ ምርትን ሊጎዳ ይችላል።",
        "Leaf diseases may reduce coffee production."
    ),
    "expert": (
        "ምልክቱ ከተባባሰ ወይም ውጤቱ እርግጠኛ ካልሆነ ባለሙያን ያማክሩ።",
        "Consult a local agricultural extension officer or qualified agricultural expert if symptoms worsen or the result is uncertain."
    ),
    "fertilizer": (
        "ከፎቶ ብቻ ማዳበሪያ መምረጥ አይቻልም። የአፈር ምርመራ ያስፈልጋል።",
        "Fertilizer cannot be safely recommended from an image alone. Soil testing is advised."
    ),
    "yellowing": (
        "ቅጠል መቢጫት በበሽታ፣ በውሃ ወይም በንጥረ ነገር እጥረት ሊከሰት ይችላል።",
        "Yellowing may result from disease, water stress, or nutrient deficiency."
    ),
}


def get_bilingual_advice(disease, intent):
    if intent in special_advice:
        return special_advice[intent]

    return (
        amharic_advice.get(
            (disease, intent),
            "ለተጨማሪ ምክር የግብርና ባለሙያን ያማክሩ።"
        ),
        english_advice.get(
            (disease, intent),
            "Consult a local agricultural extension officer or qualified agricultural expert for additional advice."
        )
    )


class_names = coffee_dataset.classes


def bunaguard_predict(img, question):
    if img is None:
        return (
            "⚠️ እባክዎ መጀመሪያ የቡና ቅጠል ፎቶ ያስገቡ።\n"
            "Please upload a coffee-leaf image first."
        )

    img = img.convert("RGB")

    quality_ok, quality_message = check_image_quality(img)
    if not quality_ok:
        return quality_message

    input_tensor = my_transform(img).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        probabilities = torch.softmax(model(input_tensor), dim=1)[0]

    top_probabilities, top_indices = torch.topk(
        probabilities,
        k=min(3, len(class_names))
    )

    disease = class_names[top_indices[0].item()]
    confidence = top_probabilities[0].item()

    if confidence < 0.60:
        return (
            "⚠️ ውጤቱ እርግጠኛ አይደለም። ፎቶውን እንደገና ያንሱ ወይም ባለሙያን ያማክሩ።\n\n"
            "The result is uncertain. Retake the image or consult a local agricultural extension officer or qualified agricultural expert."
        )

    intent = question_to_intent.get(question, "diagnosis")
    advice_amharic, advice_english = get_bilingual_advice(disease, intent)

    top_three_text = "\n".join(
        f"- {amharic_names[class_names[index.item()]]} / "
        f"{english_names[class_names[index.item()]]}: "
        f"{probability.item() * 100:.1f}%"
        for probability, index in zip(top_probabilities, top_indices)
    )

    return (
        f"🔍 ውጤት / Result: "
        f"{amharic_names[disease]} / {english_names[disease]}\n"

        f"🎯 እርግጠኝነት / Confidence: {confidence * 100:.1f}%\n\n"

        f"💬 ምክር:\n{advice_amharic}\n\n"
        f"💬 Advice:\n{advice_english}\n\n"

        f"🔢 ከፍተኛ ሶስት ግምቶች / Top Three Predictions:\n"
        f"{top_three_text}\n\n"

        "⚠️ የቡና ቅጠል ፎቶ ብቻ ያስገቡ።\n"
        "Only coffee-leaf images are supported.\n\n"

        "BunaGuard is a research prototype and does not replace expert advice. All results must be confirmed by a local agricultural extension officer or qualified agricultural expert before any treatment decision."
    )


demo = gr.Interface(
    fn=bunaguard_predict,
    inputs=[
        gr.Image(
            type="pil",
            label="📸 የቡና ቅጠል ፎቶ / Coffee-Leaf Image"
        ),
        gr.Radio(
            choices=all_questions,
            label="❓ ጥያቄ ይምረጡ / Select a Question",
            value=all_questions[0]
        )
    ],
    outputs=gr.Textbox(
        label="💬 የBunaGuard ውጤት / BunaGuard Result",
        lines=18
    ),
    title="☕ BunaGuard — የቡና በሽታ ረዳት / Coffee Leaf Disease Assistant for Ugandan Smallholder Farmers (Research Prototype)",
    description=(
        "⚠️ የቡና ቅጠል ፎቶ ብቻ ያስገቡ። "
        "BunaGuard ሌሎች ዕቃዎችን ወይም ተክሎችን መለየት አይችልም።\n\n"
        "BunaGuard is a research prototype designed to support Ugandan smallholder coffee farmers. "
        "Upload only a coffee-leaf image — the system cannot identify other objects or plants. "
        "Results must be confirmed by a local agricultural extension officer or qualified agricultural expert "
        "before any treatment decision."
    )
)

demo.launch()